<a href="https://colab.research.google.com/github/GUNJANSUMAN/accomadationfeedback/blob/main/assgn_dnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Section 1.1 - Install Required Libraries

%pip install torch numpy pandas matplotlib scikit-learn tqdm


In [2]:

# Section 1.2 - Import Required Libraries

# Basic Python libraries
import os
import random
import time
from pathlib import Path

# Data handling and numerical operations
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Evaluation metrics
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# Training progress
from tqdm.auto import tqdm

print("All required libraries imported successfully.")


All required libraries imported successfully.


In [3]:
# Section 1.3 - Check GPU and Device

import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available.")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("GPU is not available. Using CPU.")

print("PyTorch version:", torch.__version__)
print("Selected device:", device)

GPU is available.
GPU: Tesla T4
PyTorch version: 2.14.1+cu130
Selected device: cuda


In [4]:

# Section 1.4 - Assignment Configuration

# Student information
STUDENT_ID = "05160"

# Individual experiment based on the last digit of the student ID
# Even ID: RNN gradient clipping experiment
INDIVIDUAL_ABLATION = "RNN Gradient Clipping"


# Random seeds for reproducible experiments
# These are working seeds; replace them if the instructor's
# individualisation table specifies different values.
SEEDS = [42, 123, 2026]


# UCI HAR dataset parameters
INPUT_SIZE = 9
SEQUENCE_LENGTH = 128
NUM_CLASSES = 6

CLASS_NAMES = [
    "Walking",
    "Walking Upstairs",
    "Walking Downstairs",
    "Sitting",
    "Standing",
    "Laying"
]


# Recurrent neural network parameters
# Hidden size 64 is provisional for HAR until the
# assigned value in the individualisation table is confirmed.
HIDDEN_SIZE = 64

LEARNING_RATE = 0.001
BATCH_SIZE = 128
EPOCHS = 20


# Adding problem parameters
SEQUENCE_LENGTHS = [50, 100, 200, 400]

TRAIN_SAMPLES = 10000
VALIDATION_SAMPLES = 1000
TEST_SAMPLES = 1000

MSE_BASELINE = 0.167
SOLVED_THRESHOLD = 0.05


print("Assignment configuration defined successfully.")


Assignment configuration defined successfully.


In [5]:

# Section 1.5 - Random Seed Initialization

def set_seed(seed):
    # Set seed for Python random operations
    random.seed(seed)

    # Set seed for NumPy operations
    np.random.seed(seed)

    # Set seed for PyTorch CPU operations
    torch.manual_seed(seed)

    # Set seed for PyTorch GPU operations
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Initialize the notebook with the first experiment seed
set_seed(SEEDS[0])

print("Random seed initialized successfully:", SEEDS[0])


Random seed initialized successfully: 42


In [6]:

# Section 1.6 - Create Project Directories

# Set the current working directory as the project root
PROJECT_DIR = Path.cwd()

# Define directories for datasets and Problem 1 outputs
DATA_DIR = PROJECT_DIR / "data"
HAR_DATA_DIR = DATA_DIR / "UCI_HAR"

PROBLEM1_DIR = PROJECT_DIR / "problem1"
FIGURES_DIR = PROBLEM1_DIR / "figures"

# Create the required directories
for folder in [DATA_DIR, HAR_DATA_DIR, PROBLEM1_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directories created successfully.")


Project directories created successfully.


In [7]:

# Section 1.7 - Initialize Experiment Results

# Define the columns required by the assignment
RESULT_COLUMNS = [
    "problem",
    "model",
    "seed",
    "seq_len",
    "test_acc",
    "macro_f1",
    "test_mse",
    "params",
    "sec_per_epoch",
    "val_loss_nats",
    "bits_per_char",
    "notes"
]

# Set the results file location
RESULTS_FILE = PROJECT_DIR / "results.csv"

# Create an empty results file if it does not already exist
if not RESULTS_FILE.exists():
    pd.DataFrame(columns=RESULT_COLUMNS).to_csv(
        RESULTS_FILE, index=False
    )

print("Results file initialized:", RESULTS_FILE)


Results file initialized: /content/results.csv


In [8]:

# Section 1.8 - Configuration Summary

print("DNN PROGRAMMING ASSIGNMENT")
print("-" * 40)

print("Student ID       :", STUDENT_ID)
print("Computing Device :", device)
print("Random Seeds     :", SEEDS)

print("\nHAR Configuration")
print("Input Channels   :", INPUT_SIZE)
print("Sequence Length  :", SEQUENCE_LENGTH)
print("Number of Classes:", NUM_CLASSES)
print("Hidden Size      :", HIDDEN_SIZE)

print("\nAdding Problem")
print("Sequence Lengths :", SEQUENCE_LENGTHS)

print("\nIndividual Ablation")
print("Experiment       :", INDIVIDUAL_ABLATION)

print("\nResults File     :", RESULTS_FILE)

print("\nMODULE 1 COMPLETED")


DNN PROGRAMMING ASSIGNMENT
----------------------------------------
Student ID       : 05160
Computing Device : cuda
Random Seeds     : [42, 123, 2026]

HAR Configuration
Input Channels   : 9
Sequence Length  : 128
Number of Classes: 6
Hidden Size      : 64

Adding Problem
Sequence Lengths : [50, 100, 200, 400]

Individual Ablation
Experiment       : RNN Gradient Clipping

Results File     : /content/results.csv

MODULE 1 COMPLETED
